<a href="https://colab.research.google.com/github/hamdikhasawneh/AI-sepsis/blob/main/notebooks/02_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importing file (1) Data Preprocessing

In [1]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

## Load output from 01_data_preprocessing

In [2]:
import numpy as np
import pandas as pd
import zipfile
import warnings
from pathlib import Path
from scipy.stats import linregress
warnings.filterwarnings('ignore')

DATA_DIR   = Path('C:/Users/walee/OneDrive/Desktop/input path')
OUTPUT_DIR = Path('C:/Users/walee/OneDrive/Desktop/output path')

# ── Load all preprocessed outputs ───────────────────────────
print("Loading preprocessed files...")

cohort = pd.read_csv(OUTPUT_DIR / 'icu_cohort (1).csv')
cohort['intime']  = pd.to_datetime(cohort['intime'])
cohort['outtime'] = pd.to_datetime(cohort['outtime'])

vitals_complete = pd.read_csv(OUTPUT_DIR / 'vitals_complete.csv')

X = np.load(OUTPUT_DIR / 'X_vitals.npy')
print(f"X shape: {X.shape}")

stay_ids_order = (
    vitals_complete
    .sort_values(['stay_id', 'hour'])['stay_id']
    .drop_duplicates()
    .tolist()
)
print(f"Stays in X: {len(stay_ids_order):,}")

hourly_labels = pd.read_csv(OUTPUT_DIR / 'hourly_labels.csv')
hourly_labels['abs_time'] = pd.to_datetime(hourly_labels['abs_time'])
print(f"Hourly labels: {hourly_labels.shape} | "
      f"positive rate: {hourly_labels['label'].mean():.3%}")

split_df = pd.read_csv(OUTPUT_DIR / 'subject_splits.csv')

FEATURE_COLS = [
    'abp_dia', 'abp_mean', 'abp_sys',
    'heart_rate', 'resp_rate', 'spo2', 'temp_c'
]

print("\nAll files loaded ✓")
print(f"Cohort  : {cohort['stay_id'].nunique():,} stays | "
      f"{cohort['subject_id'].nunique():,} patients")
print(f"Vitals  : {vitals_complete['stay_id'].nunique():,} stays")

Loading preprocessed files...
X shape: (89075, 24, 7)
Stays in X: 89,075
Hourly labels: (2618839, 8) | positive rate: 3.332%

All files loaded ✓
Cohort  : 94,458 stays | 65,366 patients
Vitals  : 89,075 stays


## Lab based features beyond SOFA ( Lactate and WBC )
#### Lactate — elevated lactate (>2 mmol/L) is a direct sign of tissue hypoperfusion, which is a core component of septic shock. It's so important that the Sepsis-3 definition uses it to define septic shock specifically.
#### WBC (White Blood Cell count) — abnormally high OR low WBC is a classic SIRS criterion and signals the immune system responding to infection.

In [3]:
d_labitems = pd.read_csv(DATA_DIR / 'd_labitems.csv')
print(d_labitems[d_labitems['label'].str.contains('lactate', case=False, na=False)])
print(d_labitems[d_labitems['label'].str.contains('white blood', case=False, na=False)])

      itemid                           label                fluid   category
11     50813                         Lactate                Blood  Blood Gas
41     50843  Lactate Dehydrogenase, Ascites              Ascites  Chemistry
151    50954      Lactate Dehydrogenase (LD)                Blood  Chemistry
242    51054  Lactate Dehydrogenase, Pleural              Pleural  Chemistry
911    51795      Lactate Dehydrogenase, CSF  Cerebrospinal Fluid  Chemistry
1035   51944    Lactate Dehydrogenase, Stool                Stool  Chemistry
1516   52442                         Lactate                Blood  Blood Gas
1615   53154                         Lactate                Blood  Chemistry
     itemid              label  fluid    category
486   51301  White Blood Cells  Blood  Hematology
872   51755  White Blood Cells  Blood   Chemistry
873   51756  White Blood Cells  Blood   Chemistry


In [4]:
import zipfile

lab_zip_path = Path(DATA_DIR / 'chartevents_labevents.zip')
output_file  = OUTPUT_DIR / 'extra_labs_filtered.csv'

EXTRA_LAB_ITEMIDS = [50813, 52442, 51301, 50889, 51265, 51237]
# Lactate, Lactate(alt), WBC, CRP, Platelets, INR
if output_file.exists():
    output_file.unlink()

chunk_size  = 100_000
first_chunk = True
stay_ids    = set(cohort['stay_id'].dropna().unique())

with zipfile.ZipFile(lab_zip_path, 'r') as z:
    with z.open('Cleaned/labevents.csv') as f:
        for i, chunk in enumerate(
            pd.read_csv(
                f,
                chunksize=chunk_size,
                usecols=['subject_id', 'hadm_id', 'charttime', 'itemid', 'valuenum']
            )
        ):
            chunk = chunk[chunk['itemid'].isin(EXTRA_LAB_ITEMIDS)]
            chunk = chunk.merge(
                cohort[['subject_id', 'hadm_id', 'stay_id']],
                on=['subject_id', 'hadm_id'],
                how='inner'
            )
            if not chunk.empty:
                chunk.to_csv(output_file, mode='a', header=first_chunk, index=False)
                first_chunk = False
            print(f'  chunk {i+1} done', end='\r')

print('\nextra_labs_filtered.csv saved successfully')

  chunk 1584 done
extra_labs_filtered.csv saved successfully


In [5]:
extra_labs = pd.read_csv(OUTPUT_DIR / 'extra_labs_filtered.csv')
extra_labs['charttime'] = pd.to_datetime(extra_labs['charttime'])
print('Shape:', extra_labs.shape)
print('\nItemid counts:')
print(extra_labs['itemid'].value_counts())
print('\nMissing values:')
print(extra_labs.isnull().sum())
print('\nSample:')
extra_labs.head()

Shape: (4040668, 6)

Itemid counts:
itemid
51265    1363685
51301    1339779
51237     818167
50813     496975
50889      22062
Name: count, dtype: int64

Missing values:
subject_id        0
hadm_id           0
itemid            0
charttime         0
valuenum      19523
stay_id           0
dtype: int64

Sample:


,subject_id,hadm_id,itemid,charttime,valuenum,stay_id
0,10000032,29079034.0,51265,2180-07-24 06:35:00,94.0,39553978
1,10000032,29079034.0,51301,2180-07-24 06:35:00,4.1,39553978
2,10000032,29079034.0,51237,2180-07-24 06:35:00,1.8,39553978
3,10000032,29079034.0,51237,2180-07-25 04:45:00,2.0,39553978
4,10000032,29079034.0,51265,2180-07-25 04:45:00,95.0,39553978


Cleaning File

In [6]:
# Drop missing values
extra_labs = extra_labs.dropna(subset=['valuenum'])
print('After dropping nulls:', extra_labs.shape)

# Map itemids to names
extra_lab_itemid_to_name = {
    50813: 'lactate',
    52442: 'lactate',
    51301: 'wbc',
    50889: 'crp',
    51265: 'platelets',
    51237: 'inr',
}
extra_labs['lab_name'] = extra_labs['itemid'].map(extra_lab_itemid_to_name)

# Floor to hourly bins
extra_labs['charttime_hour'] = extra_labs['charttime'].dt.floor('h')

# Aggregate to hourly mean
extra_labs_hourly = (
    extra_labs
    .groupby(['stay_id', 'charttime_hour', 'lab_name'])['valuenum']
    .mean()
    .reset_index()
)

# Pivot to wide format
extra_labs_wide = extra_labs_hourly.pivot_table(
    index=['stay_id', 'charttime_hour'],
    columns='lab_name',
    values='valuenum'
).reset_index()
extra_labs_wide.columns.name = None

print('Wide format shape:', extra_labs_wide.shape)
extra_labs_wide.head()

After dropping nulls: (4021145, 6)
Wide format shape: (1766493, 7)


,stay_id,charttime_hour,crp,inr,lactate,platelets,wbc
0,30000153,2174-09-29 13:00:00,NaN,NaN,1.3,NaN,NaN
1,30000153,2174-09-29 14:00:00,NaN,NaN,2.1,NaN,NaN
2,30000153,2174-09-29 15:00:00,NaN,1.1,NaN,173.0,17.0
3,30000153,2174-09-30 03:00:00,NaN,NaN,NaN,162.0,15.2
4,30000153,2174-10-01 05:00:00,NaN,NaN,NaN,112.0,11.9


In [7]:
print(cohort.columns.tolist())
print(cohort[['stay_id', 'intime']].head())

['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'edregtime', 'edouttime', 'hospital_expire_flag', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod']
    stay_id              intime
0  39553978 2180-07-23 14:00:00
1  37081114 2150-11-02 19:37:00
2  39765666 2189-06-27 08:42:00
3  37067082 2157-11-20 19:18:02
4  34592300 2157-12-19 15:42:24


In [8]:
# Rebuild extra_labs_wide cleanly from extra_labs_hourly
extra_labs_wide = extra_labs_hourly.pivot_table(
    index=['stay_id', 'charttime_hour'],
    columns='lab_name',
    values='valuenum'
).reset_index()
extra_labs_wide.columns.name = None

# Now merge intime once
extra_labs_wide['charttime_hour'] = pd.to_datetime(extra_labs_wide['charttime_hour'])
extra_labs_wide = extra_labs_wide.merge(
    cohort[['stay_id', 'intime']],
    on='stay_id',
    how='left'
)
extra_labs_wide['intime'] = pd.to_datetime(extra_labs_wide['intime'])

# Hours since admission
extra_labs_wide['hours_since_admit'] = (
    (extra_labs_wide['charttime_hour'] - extra_labs_wide['intime'])
    .dt.total_seconds() / 3600
)

# Keep first 24 hours only
extra_labs_24h = extra_labs_wide[
    (extra_labs_wide['hours_since_admit'] >= 0) &
    (extra_labs_wide['hours_since_admit'] <  24)
].copy()

extra_labs_24h['hour'] = extra_labs_24h['hours_since_admit'].astype(int)

print('Shape after 24h filter:', extra_labs_24h.shape)
print('Stays covered:', extra_labs_24h['stay_id'].nunique())
extra_labs_24h[['stay_id', 'hour', 'lactate', 'wbc']].head(10)

Shape after 24h filter: (279165, 10)
Stays covered: 91200


,stay_id,hour,lactate,wbc
0,30000153,0,1.3,NaN
1,30000153,1,2.1,NaN
2,30000153,2,NaN,17.0
3,30000153,14,NaN,15.2
16,30000213,2,0.9,NaN
17,30000213,22,NaN,5.8
22,30000484,3,2.0,NaN
23,30000484,10,NaN,24.2
24,30000484,11,1.6,NaN
35,30000646,0,NaN,8.5


## Running remaining feature engineering sections

In [9]:
from scipy.stats import linregress

# Section 2 - Temporal vital sign features
def compute_temporal_features(vitals_complete, feature_cols):
    """
    Compute per-stay summary stats and trends over the first 24h.

    Slope fix: linregress is run ONLY on hours where the raw sensor
    actually recorded a value.  vitals_complete must include an
    'observed_{col}' boolean/int column for each vital (set to 1 before
    forward-fill, 0 on imputed rows).  If those flags are absent, the
    function falls back to all non-NaN rows — still better than running
    regression over a forward-filled series.
    """
    records = []
    for stay_id, group in vitals_complete.groupby('stay_id'):
        group = group.sort_values('hour')
        row = {'stay_id': stay_id}
        hours = group['hour'].values.astype(float)

        for col in feature_cols:
            vals = group[col].values.astype(float)
            row[f'{col}_mean'] = np.nanmean(vals)
            row[f'{col}_std']  = np.nanstd(vals)
            row[f'{col}_min']  = np.nanmin(vals)
            row[f'{col}_max']  = np.nanmax(vals)
            row[f'{col}_last'] = vals[-1] if not np.isnan(vals[-1]) else np.nanmean(vals)

            # ── Slope: use only observed (pre-imputation) rows ──────────
            obs_flag = f'observed_{col}'
            if obs_flag in group.columns:
                obs_mask = group[obs_flag].values == 1
            else:
                obs_mask = ~np.isnan(vals)      # fallback

            obs_hours = hours[obs_mask]
            obs_vals  = vals[obs_mask]

            if len(obs_vals) >= 2:
                slope, _, _, _, _ = linregress(obs_hours, obs_vals)
                row[f'{col}_slope'] = slope
            else:
                row[f'{col}_slope'] = np.nan    # not enough real readings

        records.append(row)
    return pd.DataFrame(records)

print('Computing temporal features — this will take a few minutes...')
temporal_features = compute_temporal_features(vitals_complete, FEATURE_COLS)
print(f'Temporal features shape: {temporal_features.shape}')


Computing temporal features — this will take a few minutes...
Temporal features shape: (89075, 43)


In [11]:
# Section 3 - SOFA summary features
sofa_labs = pd.read_csv(OUTPUT_DIR / 'sofa_labs_hourly_wide.csv')
sofa_labs['charttime_hour'] = pd.to_datetime(sofa_labs['charttime_hour'])

def compute_sofa_features(sofa_labs, cohort):
    sofa = sofa_labs.merge(
        cohort[['stay_id', 'intime']], on='stay_id', how='left'
    )
    sofa['intime'] = pd.to_datetime(sofa['intime'])
    sofa['charttime_hour'] = pd.to_datetime(sofa['charttime_hour'])
    sofa['hours_since_admit'] = (
        (sofa['charttime_hour'] - sofa['intime']).dt.total_seconds() / 3600
    )
    sofa_24h = sofa[
        (sofa['hours_since_admit'] >= 0) &
        (sofa['hours_since_admit'] <  24)
    ].copy()
    sofa_24h['hour'] = sofa_24h['hours_since_admit'].astype(int)

    # Recompute all SOFA lab scores from raw values
    def score_platelets(x):
        if pd.isna(x): return np.nan
        if x >= 150:   return 0
        if x >= 100:   return 1
        if x >= 50:    return 2
        if x >= 20:    return 3
        return 4

    def score_bilirubin(x):
        if pd.isna(x): return np.nan
        if x < 1.2:    return 0
        if x < 2.0:    return 1
        if x < 6.0:    return 2
        if x < 12.0:   return 3
        return 4

    def score_creatinine(x):
        if pd.isna(x): return np.nan
        if x < 1.2:    return 0
        if x < 2.0:    return 1
        if x < 3.5:    return 2
        if x < 5.0:    return 3
        return 4

    sofa_24h['sofa_platelets']  = sofa_24h['platelets'].apply(score_platelets)
    sofa_24h['sofa_bilirubin']  = sofa_24h['bilirubin'].apply(score_bilirubin)
    sofa_24h['sofa_creatinine'] = sofa_24h['creatinine'].apply(score_creatinine)
    sofa_24h['sofa_lab_total']  = sofa_24h[
        ['sofa_platelets', 'sofa_bilirubin', 'sofa_creatinine']
    ].sum(axis=1, min_count=1)

    records = []
    for stay_id, group in sofa_24h.groupby('stay_id'):
        group = group.sort_values('hour')
        vals  = group['sofa_lab_total'].values.astype(float)
        h     = group['hour'].values
        mask  = ~np.isnan(vals)

        slope = np.nan
        if mask.sum() >= 2:
            slope, *_ = linregress(h[mask], vals[mask])

        records.append({
            'stay_id':        stay_id,
            'sofa_max_24h':   np.nanmax(vals)  if mask.any() else np.nan,
            'sofa_mean_24h':  np.nanmean(vals) if mask.any() else np.nan,
            'sofa_last_24h':  vals[-1] if not np.isnan(vals[-1])
                              else np.nanmean(vals),
            'sofa_slope_24h': slope,
            'sofa_hours_ge2': int((vals >= 2).sum()),
        })

    return pd.DataFrame(records)

print('Computing SOFA features...')
sofa_features = compute_sofa_features(sofa_labs, cohort)
print(f'SOFA features shape: {sofa_features.shape}')
print(sofa_features.describe())

Computing SOFA features...
SOFA features shape: (90562, 6)
            stay_id  sofa_max_24h  sofa_mean_24h  sofa_last_24h  \
count  9.056200e+04  90562.000000   90562.000000   90562.000000   
mean   3.499763e+07      1.431759       1.119263       1.114220   
std    2.884840e+06      1.757002       1.416626       1.509087   
min    3.000015e+07      0.000000       0.000000       0.000000   
25%    3.250923e+07      0.000000       0.000000       0.000000   
50%    3.499987e+07      1.000000       0.666667       1.000000   
75%    3.748774e+07      2.000000       2.000000       2.000000   
max    3.999986e+07     12.000000      11.000000      12.000000   

       sofa_slope_24h  sofa_hours_ge2  
count    62937.000000    90562.000000  
mean        -0.004185        0.801418  
std          0.140002        1.368955  
min         -7.000000        0.000000  
25%         -0.004348        0.000000  
50%          0.000000        0.000000  
75%          0.000000        1.000000  
max          6.00

# Static Patients Features

In [12]:
# Section 4 - Static patient features
def compute_static_features(cohort):
    """
    Build static features from cohort demographics.
    icu_los_hours is excluded — future leakage.
    """
    static = cohort[[
        'stay_id', 'anchor_age', 'gender',
        'admission_type', 'admission_location',
    ]].copy()

    # Binary gender encoding
    static['gender_male'] = (static['gender'].str.upper() == 'M').astype(int)

    # One-hot encode admission type
    admission_dummies = pd.get_dummies(
        static['admission_type'].str.lower().str.replace(' ', '_'),
        prefix='adm_type'
    ).astype(int)
    static = pd.concat([static, admission_dummies], axis=1)

    # Drop original string columns
    static = static.drop(
        columns=['gender', 'admission_type', 'admission_location']
    )

    return static

static_features = compute_static_features(cohort)
print(f'Static features shape: {static_features.shape}')
print('\nColumns:', static_features.columns.tolist())
static_features.head()

Static features shape: (94458, 12)

Columns: ['stay_id', 'anchor_age', 'gender_male', 'adm_type_ambulatory_observation', 'adm_type_direct_emer.', 'adm_type_direct_observation', 'adm_type_elective', 'adm_type_eu_observation', 'adm_type_ew_emer.', 'adm_type_observation_admit', 'adm_type_surgical_same_day_admission', 'adm_type_urgent']


,stay_id,anchor_age,gender_male,adm_type_ambulatory_observation,adm_type_direct_emer.,adm_type_direct_observation,adm_type_elective,adm_type_eu_observation,adm_type_ew_emer.,adm_type_observation_admit,adm_type_surgical_same_day_admission,adm_type_urgent
0,39553978,52,0,0,0,0,0,0,1,0,0,0
1,37081114,86,0,0,0,0,0,0,1,0,0,0
2,39765666,73,0,0,0,0,0,0,1,0,0,0
3,37067082,55,0,0,0,0,0,0,1,0,0,0
4,34592300,55,0,0,1,0,0,0,0,0,0,0


In [13]:
# Section 5 - Missingness indicators
def compute_missingness_features(vitals_complete, feature_cols):
    records = []
    for stay_id, group in vitals_complete.groupby('stay_id'):
        row = {'stay_id': stay_id}
        for col in feature_cols:
            n_missing = group[col].isna().sum()
            row[f'{col}_missing_frac'] = n_missing / len(group)
        records.append(row)
    return pd.DataFrame(records)
    
def compute_lab_missingness(extra_labs_24h):
    """Binary flag: was this lab measured at all in first 24h?"""
    records = []
    for stay_id, group in extra_labs_24h.groupby('stay_id'):
        records.append({
            'stay_id':              stay_id,
            'crp_observed_24h':     int(group['crp'].notna().any()),
            'platelets_observed_24h': int(group['platelets'].notna().any()),
            'inr_observed_24h':     int(group['inr'].notna().any()),
        })
    return pd.DataFrame(records)

lab_missingness = compute_lab_missingness(extra_labs_24h)
print('Computing missingness features...')
missingness_features = compute_missingness_features(vitals_complete, FEATURE_COLS)
print(f'Missingness features shape: {missingness_features.shape}')
missingness_features.head()

Computing missingness features...
Missingness features shape: (89075, 8)


,stay_id,abp_dia_missing_frac,abp_mean_missing_frac,abp_sys_missing_frac,heart_rate_missing_frac,resp_rate_missing_frac,spo2_missing_frac,temp_c_missing_frac
0,30000153,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,30000213,1.0,1.0,1.0,0.0,0.0,0.0,1.0
2,30000484,1.0,1.0,1.0,0.0,0.0,0.0,1.0
3,30000646,1.0,1.0,1.0,0.0,0.0,0.0,1.0
4,30000831,1.0,1.0,1.0,0.0,0.0,0.0,1.0


## Forward fills labs

In [14]:
# Carry forward last known lactate and WBC within each stay
extra_labs_24h = extra_labs_24h.sort_values(['stay_id', 'hour'])

extra_labs_24h['lactate'] = (
    extra_labs_24h.groupby('stay_id')['lactate']
    .transform(lambda x: x.ffill())
)

extra_labs_24h['wbc'] = (
    extra_labs_24h.groupby('stay_id')['wbc']
    .transform(lambda x: x.ffill())
)

for lab in ['lactate', 'wbc', 'crp', 'platelets', 'inr']:
    extra_labs_24h[lab] = (
        extra_labs_24h.groupby('stay_id')[lab]
        .transform(lambda x: x.ffill())
    )

print('After forward fill:')
print(extra_labs_24h[['stay_id', 'hour', 'lactate', 'wbc']].head(15))
print('\nRemaining NaN:')
print(extra_labs_24h[['lactate', 'wbc']].isnull().sum())

After forward fill:
     stay_id  hour  lactate   wbc
0   30000153     0      1.3   NaN
1   30000153     1      2.1   NaN
2   30000153     2      2.1  17.0
3   30000153    14      2.1  15.2
16  30000213     2      0.9   NaN
17  30000213    22      0.9   5.8
22  30000484     3      2.0   NaN
23  30000484    10      2.0  24.2
24  30000484    11      1.6  24.2
35  30000646     0      NaN   8.5
36  30000646     4      1.6  10.6
37  30000646     8      1.6  10.6
38  30000646    17      1.6   7.9
48  30000831     7      1.4  14.2
49  30000831    14      1.4  14.2

Remaining NaN:
lactate    85557
wbc        29274
dtype: int64


## Computing Lab features

In [15]:
import numpy as np

def compute_lab_features(extra_labs_24h: pd.DataFrame) -> pd.DataFrame:
    """
    For each stay compute summary statistics for lactate and WBC
    over the first 24 hours.
    Slope is computed only on observed (non-imputed) values.
    """
    records = []

    for stay_id, group in extra_labs_24h.groupby('stay_id'):
        row = {'stay_id': stay_id}

        for lab in ['lactate', 'wbc', 'crp', 'platelets', 'inr']:
            vals = group[lab].dropna().values

            if len(vals) == 0:
                row[f'{lab}_mean']    = np.nan
                row[f'{lab}_max']     = np.nan
                row[f'{lab}_min']     = np.nan
                row[f'{lab}_first']   = np.nan
                row[f'{lab}_last']    = np.nan
                row[f'{lab}_count']   = 0
                row[f'{lab}_missing'] = 1
                row[f'{lab}_slope']   = np.nan
            else:
                row[f'{lab}_mean']    = np.nanmean(vals)
                row[f'{lab}_max']     = np.nanmax(vals)
                row[f'{lab}_min']     = np.nanmin(vals)
                row[f'{lab}_first']   = vals[0]
                row[f'{lab}_last']    = vals[-1]
                row[f'{lab}_count']   = len(vals)
                row[f'{lab}_missing'] = 0

                # Slope only on observed rows
                obs_rows  = group[group[lab].notna()]
                obs_vals  = obs_rows[lab].values.astype(float)
                obs_hours = obs_rows['hour'].values.astype(float)

                if len(obs_vals) >= 2:
                    from scipy.stats import linregress
                    slope, *_ = linregress(obs_hours, obs_vals)
                    row[f'{lab}_slope'] = slope
                else:
                    row[f'{lab}_slope'] = np.nan


        # Clinical threshold flags
        lactate_vals = group['lactate'].dropna().values
        wbc_vals     = group['wbc'].dropna().values
        plt_vals = group['platelets'].dropna().values
        inr_vals = group['inr'].dropna().values
        crp_vals = group['crp'].dropna().values

        row['platelets_low']      = int(any(plt_vals < 150))   # SOFA threshold
        row['platelets_critical'] = int(any(plt_vals < 50))    # critical threshold
        row['inr_elevated']       = int(any(inr_vals > 1.5))   # coagulopathy marker
        row['crp_elevated']       = int(any(crp_vals > 10.0))  # standard cutoff mg/L
        row['crp_missing']        = int(len(crp_vals) == 0)    # CRP often absent — flag it
        

        row['lactate_elevated'] = int(any(lactate_vals > 2.0))
        row['lactate_critical'] = int(any(lactate_vals > 4.0))
        row['wbc_high']         = int(any(wbc_vals > 12.0))
        row['wbc_low']          = int(any(wbc_vals < 4.0))
        row['wbc_abnormal']     = int(
            any(wbc_vals > 12.0) or any(wbc_vals < 4.0)
        )

        records.append(row)

    return pd.DataFrame(records)

print('compute_lab_features defined ✓')

compute_lab_features defined ✓


## Recompute lab features

In [16]:
# Recompute lab features with forward-filled data
print('Recomputing lab features with forward-filled values...')
lab_features = compute_lab_features(extra_labs_24h)
print(f'Lab features shape: {lab_features.shape}')
print('\nMissing values:')
print(lab_features.isnull().sum())

Recomputing lab features with forward-filled values...
Lab features shape: (91200, 50)

Missing values:
stay_id                   0
lactate_mean          42018
lactate_max           42018
lactate_min           42018
lactate_first         42018
lactate_last          42018
lactate_count             0
lactate_missing           0
lactate_slope         48161
wbc_mean               1201
wbc_max                1201
wbc_min                1201
wbc_first              1201
wbc_last               1201
wbc_count                 0
wbc_missing               0
wbc_slope             32379
crp_mean              88977
crp_max               88977
crp_min               88977
crp_first             88977
crp_last              88977
crp_count                 0
crp_missing               0
crp_slope             89948
platelets_mean         1186
platelets_max          1186
platelets_min          1186
platelets_first        1186
platelets_last         1186
platelets_count           0
platelets_missing         0


## Combining final features into one table

In [17]:
# Section 6 - Combine all features (labels kept separate)
all_features = (
    pd.DataFrame({'stay_id': stay_ids_order})
    .merge(temporal_features,    on='stay_id', how='left')
    .merge(sofa_features,        on='stay_id', how='left')
    .merge(static_features,      on='stay_id', how='left')
    .merge(missingness_features, on='stay_id', how='left')
    .merge(lab_features,         on='stay_id', how='left')
    .merge(lab_missingness,      on='stay_id', how='left')   # ← add this line
)

print(f'Feature table shape: {all_features.shape}')
print(f'Total features: {all_features.shape[1] - 1}')
print(f'\nMissing values (top 15):')
print(all_features.isnull().sum().sort_values(ascending=False).head(15))

Feature table shape: (89075, 118)
Total features: 117

Missing values (top 15):
crp_slope        87869
crp_mean         86951
crp_min          86951
crp_max          86951
crp_first        86951
crp_last         86951
temp_c_slope     81711
temp_c_min       81711
temp_c_std       81711
temp_c_last      81711
temp_c_max       81711
temp_c_mean      81711
abp_sys_slope    57499
abp_sys_min      57499
abp_sys_max      57499
dtype: int64


## Handeling missingness

In [18]:
print(split_df.columns.tolist())
print(split_df.head(2))

['subject_id', 'split']
   subject_id  split
0    15904420  train
1    16309699  train


In [19]:
# Section 7 - Fill remaining NaN with train-only median
feature_cols_all = [c for c in all_features.columns if c != 'stay_id']

# Map subject-level splits to stay-level via cohort
stay_to_split = cohort[['stay_id', 'subject_id']].merge(split_df, on='subject_id', how='left')
train_stay_ids = set(stay_to_split[stay_to_split['split'] == 'train']['stay_id'])

# Compute medians on training stays only
train_mask = all_features['stay_id'].isin(train_stay_ids)
print(f"Training stays for median computation: {train_mask.sum():,}")

medians = all_features.loc[train_mask, feature_cols_all].median()
all_features[feature_cols_all] = all_features[feature_cols_all].fillna(medians)

print('Missing values after median fill:', all_features.isnull().sum().sum())
print(f'\nFinal shape: {all_features.shape}')
print(f'Total features (excl. stay_id): {len(feature_cols_all)}')

Training stays for median computation: 55,237
Missing values after median fill: 0

Final shape: (89075, 118)
Total features (excl. stay_id): 117


In [20]:
# Section 8 - Save outputs

# Feature table (no labels)
all_features.to_csv(OUTPUT_DIR / 'engineered_features.csv', index=False)
print(f'Saved engineered_features.csv — shape: {all_features.shape}')

# Medians for inference-time imputation
medians.to_csv(OUTPUT_DIR / 'feature_medians.csv', header=True)
print('Saved feature_medians.csv')

# LSTM array
np.save(OUTPUT_DIR / 'X_vitals.npy', X)
print(f'Saved X_vitals.npy — shape: {X.shape}')

# Feature column names
feature_cols_all = [c for c in all_features.columns if c != 'stay_id']
with open(OUTPUT_DIR / 'feature_names.txt', 'w') as f:
    f.write('\n'.join(feature_cols_all))
print(f'Saved feature_names.txt — {len(feature_cols_all)} features')

# Leakage check
leaked = [c for c in all_features.columns
          if c in ['y_6h', 'y_12h', 'y_24h',
                   'eligible_6h', 'eligible_12h', 'eligible_24h',
                   'icu_los_hours']]
assert len(leaked) == 0, f'LEAKAGE: {leaked}'
print('Leakage check passed ✓')

Saved engineered_features.csv — shape: (89075, 118)
Saved feature_medians.csv
Saved X_vitals.npy — shape: (89075, 24, 7)
Saved feature_names.txt — 117 features
Leakage check passed ✓


## Summary of Labels

In [21]:
# Label summary from rolling hourly labels
print('\n--- Label Summary ---')
print(f"Total prediction points : {len(hourly_labels):,}")
print(f"Positive (sepsis)       : {hourly_labels['label'].sum():,}")
print(f"Positive rate           : {hourly_labels['label'].mean():.3%}")
for split in ['train', 'val', 'test']:
    s = hourly_labels[hourly_labels['split'] == split]
    print(f"{split:5s} | {len(s):>9,} points | "
          f"positive rate {s['label'].mean():.3%}")


--- Label Summary ---
Total prediction points : 2,618,839
Positive (sepsis)       : 87,263
Positive rate           : 3.332%
train | 1,822,797 points | positive rate 3.384%
val   |   396,334 points | positive rate 3.149%
test  |   399,708 points | positive rate 3.277%


## Gap Features: Urine Output, Vasopressor Flag, Ventilation Status, Blood Culture

These four features appear in EPIC Sepsis Model and Sepsis Watch but were absent.
They are extracted here and merged into `all_features` before saving.
Run this block after Section 8 above re-saves `engineered_features.csv`.


###### ── FIX (issue #4): Missing features from published models ───────────────────
#
######  1. Urine output (outputevents)      — organ dysfunction signal
######  2. Vasopressor flag (inputevents)   — already extracted in preprocessing
######     if vasopressors_filtered.csv is present; binary flag here
######  3. Ventilation status (chartevents) — procedureevents / ventilation table
######  4. Blood culture result             — microbiologyevents (positive flag)
###### All four are joined to all_features and the combined CSV is re-saved.
######─────────────────────────────────────────────────

In [24]:
# ── 1. URINE OUTPUT ─────────────────────────────────────────────────────────
uo_output = OUTPUT_DIR / 'urine_output_filtered.csv'
URINE_ITEMIDS = [
    226559, 226560, 226561, 226584, 226563, 226564, 226565, 226567,
    226557, 226558, 227488, 227489,
]
stay_ids = set(cohort['stay_id'].dropna().unique())

if not uo_output.exists():
    first_chunk = True
    with open(DATA_DIR / 'outputevents.csv') as f:
        for i, chunk in enumerate(pd.read_csv(
            f, chunksize=100_000,
            usecols=['stay_id', 'charttime', 'itemid', 'value']
        )):
            chunk = chunk[
                chunk['stay_id'].isin(stay_ids) &
                chunk['itemid'].isin(URINE_ITEMIDS)
            ]
            if not chunk.empty:
                chunk.to_csv(uo_output, mode='a',
                             header=first_chunk, index=False)
                first_chunk = False
        print(f'UO chunk {i+1}', end='\r')
    print('\nurine_output_filtered.csv saved')
else:
    print('urine_output_filtered.csv already exists, skipping extraction')

uo_df = pd.read_csv(uo_output)
uo_df['charttime'] = pd.to_datetime(uo_df['charttime'])
uo_df = uo_df.merge(cohort[['stay_id', 'intime']], on='stay_id', how='left')
uo_df['intime'] = pd.to_datetime(uo_df['intime'])
uo_df['hours_since_admit'] = (
    (uo_df['charttime'] - uo_df['intime']).dt.total_seconds() / 3600
)
uo_24h = uo_df[
    (uo_df['hours_since_admit'] >= 0) &
    (uo_df['hours_since_admit'] <  24)
].copy()
uo_24h['value'] = pd.to_numeric(uo_24h['value'], errors='coerce')
# Clip physiologically impossible values
# Negative UO = data correction entries, remove them
# Max realistic 24h UO is ~10,000 mL even with aggressive diuresis
uo_24h['value'] = uo_24h['value'].clip(lower=0, upper=10000)
uo_features = (
    uo_24h.groupby('stay_id')['value']
    .agg(
        uo_total_24h='sum',
        uo_mean_hourly=lambda x: x.sum() / 24,
        uo_min_hourly='min',
        uo_count='count'
    )
    .reset_index()
)
# Clinical flag: oliguria = total UO < 500 mL in 24h
uo_features['oliguria_flag'] = (
    uo_features['uo_total_24h'] < 500
).astype(int)

print(f'UO features shape: {uo_features.shape}')
print(uo_features.describe())

# ── 2. VASOPRESSOR FLAG ──────────────────────────────────────────────────────
# vasopressors_filtered.csv already extracted in preprocessing (Block 6)
vaso_path = OUTPUT_DIR / 'vasopressors_filtered.csv'

if vaso_path.exists():
    vaso_df = pd.read_csv(vaso_path)
    vaso_df['starttime']      = pd.to_datetime(vaso_df['starttime'])
    vaso_df['charttime_hour'] = vaso_df['starttime'].dt.floor('h')
    vaso_df = vaso_df.merge(
        cohort[['stay_id', 'intime']], on='stay_id', how='left'
    )
    vaso_df['intime'] = pd.to_datetime(vaso_df['intime'])
    vaso_df['hours_since_admit'] = (
        (vaso_df['charttime_hour'] - vaso_df['intime'])
        .dt.total_seconds() / 3600
    )
    vaso_24h = vaso_df[
        (vaso_df['hours_since_admit'] >= 0) &
        (vaso_df['hours_since_admit'] <  24)
    ].copy()

    vaso_features = (
        vaso_24h.groupby('stay_id')
        .size()
        .reset_index(name='vaso_events_24h')
    )
    vaso_features['vasopressor_flag'] = 1
    print(f'Vasopressor features: {vaso_features.shape}')
else:
    print('WARNING: vasopressors_filtered.csv not found')
    vaso_features = pd.DataFrame(columns=['stay_id', 'vaso_events_24h',
                                           'vasopressor_flag'])

# ── 4. BLOOD CULTURE FLAG ────────────────────────────────────────────────────
# Blood culture drawn and positive result flags from microbiologyevents
micro = pd.read_csv(DATA_DIR / 'microbiologyevents.csv', low_memory=False)
micro['charttime'] = pd.to_datetime(micro['charttime'], errors='coerce')
micro['chartdate'] = pd.to_datetime(micro['chartdate'], errors='coerce')
micro['culture_time'] = micro['charttime'].combine_first(micro['chartdate'])

cohort_keys = cohort[['subject_id', 'hadm_id', 'stay_id',
                       'intime']].drop_duplicates()
micro = micro.merge(cohort_keys, on=['subject_id', 'hadm_id'], how='inner')
micro['intime'] = pd.to_datetime(micro['intime'])
micro['hours_since_admit'] = (
    (micro['culture_time'] - micro['intime']).dt.total_seconds() / 3600
)

# Blood cultures only, within first 24h
blood_cultures = micro[
    (micro['spec_type_desc'].str.contains('BLOOD', case=False, na=False)) &
    (micro['hours_since_admit'] >= 0) &
    (micro['hours_since_admit'] <  24)
].copy()

# Drawn flag
culture_drawn = (
    blood_cultures.groupby('stay_id')
    .size()
    .reset_index(name='culture_count')
)
culture_drawn['blood_culture_drawn'] = 1

# Positive flag — any organism grown
culture_positive = blood_cultures[
    blood_cultures['org_name'].notna() &
    (blood_cultures['org_name'].str.strip() != '')
][['stay_id']].drop_duplicates()
culture_positive['blood_culture_positive'] = 1

blood_culture_features = culture_drawn[
    ['stay_id', 'blood_culture_drawn']
].merge(
    culture_positive, on='stay_id', how='left'
)
blood_culture_features['blood_culture_positive'] = (
    blood_culture_features['blood_culture_positive'].fillna(0).astype(int)
)

print(f'Blood culture features shape: {blood_culture_features.shape}')
print(f'Cultures drawn: {blood_culture_features["blood_culture_drawn"].sum():,}')
print(f'Cultures positive: {blood_culture_features["blood_culture_positive"].sum():,}')

urine_output_filtered.csv already exists, skipping extraction
UO features shape: (89442, 6)
            stay_id   uo_total_24h  uo_mean_hourly  uo_min_hourly  \
count  8.944200e+04   89442.000000    89442.000000   89442.000000   
mean   3.499860e+07    1780.715374       74.196474      97.645018   
std    2.887061e+06    2476.417629      103.184068     149.444685   
min    3.000015e+07       0.000000        0.000000       0.000000   
25%    3.250736e+07     880.000000       36.666667      20.000000   
50%    3.500046e+07    1475.000000       61.458333      40.000000   
75%    3.749118e+07    2280.000000       95.000000     100.000000   
max    3.999986e+07  199750.000000     8322.916667    3000.000000   

           uo_count  oliguria_flag  
count  89442.000000   89442.000000  
mean      12.321214       0.113739  
std        7.201116       0.317495  
min        1.000000       0.000000  
25%        5.000000       0.000000  
50%       13.000000       0.000000  
75%       19.000000       0

In [25]:
# ── 3. VENTILATION STATUS ────────────────────────────────────────────────────
# Try ventilation.csv first, fall back to procedureevents
vent_path      = DATA_DIR / 'ventilation.csv'
vent_proc_path = DATA_DIR / 'procedureevents.csv'
VENT_PROC_ITEMIDS = [225792, 225794]  # Invasive/non-invasive ventilation

if vent_path.exists():
    vent_raw = pd.read_csv(
        vent_path,
        usecols=['stay_id', 'starttime', 'ventilation_status']
    )
    vent_raw['starttime'] = pd.to_datetime(vent_raw['starttime'])
    vent_raw = vent_raw.merge(
        cohort[['stay_id', 'intime']], on='stay_id', how='left'
    )
    vent_raw['intime'] = pd.to_datetime(vent_raw['intime'])
    vent_raw['hours_since_admit'] = (
        (vent_raw['starttime'] - vent_raw['intime'])
        .dt.total_seconds() / 3600
    )
    vent_24h = vent_raw[
        (vent_raw['hours_since_admit'] >= 0) &
        (vent_raw['hours_since_admit'] <  24) &
        (vent_raw['ventilation_status'] == 'InvasiveVent')
    ]
    print('Ventilation data loaded from ventilation.csv')

elif vent_proc_path.exists():
    vent_raw = pd.read_csv(
        vent_proc_path,
        usecols=['stay_id', 'starttime', 'itemid']
    )
    vent_raw = vent_raw[vent_raw['itemid'].isin(VENT_PROC_ITEMIDS)]
    vent_raw['starttime'] = pd.to_datetime(vent_raw['starttime'])
    vent_raw = vent_raw.merge(
        cohort[['stay_id', 'intime']], on='stay_id', how='left'
    )
    vent_raw['intime'] = pd.to_datetime(vent_raw['intime'])
    vent_raw['hours_since_admit'] = (
        (vent_raw['starttime'] - vent_raw['intime'])
        .dt.total_seconds() / 3600
    )
    vent_24h = vent_raw[
        (vent_raw['hours_since_admit'] >= 0) &
        (vent_raw['hours_since_admit'] <  24)
    ]
    print('Ventilation data loaded from procedureevents.csv')

else:
    print('WARNING: No ventilation source found — flag will be all zeros')
    vent_24h = pd.DataFrame(columns=['stay_id'])

ventilated_stays = (
    set(vent_24h['stay_id'].unique()) if len(vent_24h) > 0 else set()
)
vent_features = cohort[['stay_id']].copy()
vent_features['ventilated_flag'] = (
    vent_features['stay_id'].isin(ventilated_stays).astype(int)
)

print(f'Ventilation features shape: {vent_features.shape}')
print(f'Ventilated stays in first 24h: {vent_features["ventilated_flag"].sum():,}')

# ── Merge all gap features into all_features ─────────────────────────────────
all_features = all_features.merge(
    uo_features, on='stay_id', how='left'
).merge(
    vaso_features[['stay_id', 'vaso_events_24h', 'vasopressor_flag']],
    on='stay_id', how='left'
).merge(
    vent_features, on='stay_id', how='left'
).merge(
    blood_culture_features, on='stay_id', how='left'
)

# Fill binary flags with 0 for stays with no event
for col in ['vasopressor_flag', 'blood_culture_drawn',
            'blood_culture_positive', 'vaso_events_24h']:
    if col in all_features.columns:
        all_features[col] = all_features[col].fillna(0).astype(int)

print(f'\nall_features shape after gap features: {all_features.shape}')
print(f'Total features: {all_features.shape[1] - 1}')

Ventilation features shape: (94458, 2)
Ventilated stays in first 24h: 0

all_features shape after gap features: (89075, 128)
Total features: 127


In [26]:
# Median-fill the new columns
new_gap_cols = [
    'uo_total_24h', 'uo_mean_hourly', 'uo_min_hourly', 'uo_count', 'oliguria_flag',
    'vaso_events_24h', 'vasopressor_flag', 'ventilated_flag',
    'blood_culture_drawn', 'blood_culture_positive'
]
new_gap_cols_present = [c for c in new_gap_cols if c in all_features.columns]
gap_medians = all_features[new_gap_cols_present].median()
all_features[new_gap_cols_present] = all_features[new_gap_cols_present].fillna(gap_medians)

# Re-save
all_features.to_csv(OUTPUT_DIR / 'engineered_features.csv', index=False)

feature_cols_all = [c for c in all_features.columns if c != 'stay_id']
with open(OUTPUT_DIR / 'feature_names.txt', 'w') as f:
    f.write('\n'.join(feature_cols_all))

print(f'\nUpdated engineered_features.csv — shape: {all_features.shape}')
print(f'Total features (excl. stay_id): {len(feature_cols_all)}')
print(f'\nNew gap features added:')
for c in new_gap_cols_present:
    print(f'  {c}: {all_features[c].notna().sum():,} non-null')



Updated engineered_features.csv — shape: (89075, 128)
Total features (excl. stay_id): 127

New gap features added:
  uo_total_24h: 89,075 non-null
  uo_mean_hourly: 89,075 non-null
  uo_min_hourly: 89,075 non-null
  uo_count: 89,075 non-null
  oliguria_flag: 89,075 non-null
  vaso_events_24h: 89,075 non-null
  vasopressor_flag: 89,075 non-null
  ventilated_flag: 89,075 non-null
  blood_culture_drawn: 89,075 non-null
  blood_culture_positive: 89,075 non-null


## Verification step

In [27]:
# Final verification
print("=" * 50)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 50)
print(f"Shape              : {all_features.shape}")
print(f"Total features     : {all_features.shape[1] - 1}")
print(f"Missing values     : {all_features.isnull().sum().sum()}")
print(f"Unique stays       : {all_features['stay_id'].nunique():,}")

print("\nFeature groups:")
print(f"  Temporal vitals  : {len([c for c in all_features.columns if any(v in c for v in ['heart_rate','resp_rate','temp_c','spo2','abp']) and c != 'stay_id']):,}")
print(f"  SOFA summary     : {len([c for c in all_features.columns if 'sofa' in c]):,}")
print(f"  Static           : {len([c for c in all_features.columns if 'age' in c or 'gender' in c or 'adm_type' in c]):,}")
print(f"  Missingness      : {len([c for c in all_features.columns if 'missing' in c and 'lac' not in c and 'wbc' not in c]):,}")
print(f"  Lactate/WBC      : {len([c for c in all_features.columns if 'lactate' in c or 'wbc' in c]):,}")
print(f"  Gap features     : {len([c for c in all_features.columns if any(g in c for g in ['uo_','vaso','vent','culture'])]):,}")

print("\nLeakage check:")
leaked = [c for c in all_features.columns
          if c in ['y_6h','y_12h','y_24h','eligible_6h',
                   'eligible_12h','eligible_24h','icu_los_hours']]
print(f"  Leaked columns   : {leaked if leaked else 'None ✓'}")

print("\nFiles saved to Drive:")
for fname in ['engineered_features.csv', 'feature_medians.csv',
              'X_vitals.npy', 'feature_names.txt']:
    fpath = OUTPUT_DIR / fname
    if fpath.exists():
        size_mb = fpath.stat().st_size / 1_048_576
        print(f"  {fname:<35s} {size_mb:>7.1f} MB")
    else:
        print(f"  {fname:<35s} NOT FOUND")

FEATURE ENGINEERING COMPLETE
Shape              : (89075, 128)
Total features     : 127
Missing values     : 0
Unique stays       : 89,075

Feature groups:
  Temporal vitals  : 49
  SOFA summary     : 5
  Static           : 11
  Missingness      : 10
  Lactate/WBC      : 21
  Gap features     : 9

Leakage check:
  Leaked columns   : None ✓

Files saved to Drive:
  engineered_features.csv                79.7 MB
  feature_medians.csv                     0.0 MB
  X_vitals.npy                          114.2 MB
  feature_names.txt                       0.0 MB
